# Sentiment Analysis — TF-IDF + Logistic Regression Baseline vs. Lexicon-Based Advanced Model

**Project:** Task 3 — Sentiment Analysis (Beginner–Intermediate)

**⚠️ Offline sandbox notice:** This environment has no internet access. That affects this project in two ways, both documented inline as they occur:
1. **Dataset:** No Twitter/Reddit/IMDB dataset could be downloaded (no Kaggle API, no `nltk.download()`). A synthetic labeled review dataset is generated instead (`scripts/generate_dataset.py`), designed to have realistic difficulty (negation, mixed-sentiment noise) rather than trivially separable classes.
2. **Advanced model:** `transformers` cannot be installed (no PyPI access) and no pretrained Hugging Face weights can be downloaded. A lexicon + negation-aware rule-based scorer is used as a documented substitute for the 'Advanced' tier, playing the same role an off-the-shelf pretrained model would (evaluated without training on this dataset's labels). The equivalent Hugging Face code is included as a comment for when network access is available.

## Contents
1. Dataset generation & class balance
2. Text cleaning / tokenization (custom, no nltk)
3. Baseline: TF-IDF + Logistic Regression (+ SVM, Naive Bayes comparisons)
4. Advanced substitute: Lexicon rule-based scorer
5. Metrics table & confusion matrices
6. Saved model + inference demo


## 1. Dataset generation & class balance

See `scripts/generate_dataset.py` for full generation logic and the offline-substitution rationale.

In [1]:
import sys
sys.path.insert(0, '../scripts')
import pandas as pd

df = pd.read_csv('../data/reviews.csv')
print(df.shape)
df['label'].value_counts()

(925, 2)


label
positive    500
negative    425
Name: count, dtype: int64

In [2]:
df.head(8)

### Text length distribution (quick EDA)

In [3]:
import matplotlib.pyplot as plt
df['n_words'] = df['text'].str.split().str.len()
df['n_words'].hist(bins=20)
plt.xlabel('Word count per review'); plt.ylabel('Frequency')
plt.title('Review length distribution')
plt.show()

## 2. Text cleaning & tokenization

`nltk` is unavailable offline, so `scripts/preprocess.py` reimplements the two pieces this project needs (regex tokenizer + hand-compiled stopword list) using only the standard library. Functionally equivalent to `nltk.word_tokenize()` + `nltk.corpus.stopwords.words('english')` for this task.

In [4]:
from preprocess import preprocess_for_tfidf
sample = df['text'].iloc[0]
print('Original:', sample)
print('Cleaned :', preprocess_for_tfidf(sample))

Original: This movie five stars, no complaints at all, the staff were incredibly friendly.
Cleaned : movie five stars complaints staff incredibly friendly


## 3. Baseline: TF-IDF + Logistic Regression (with SVM & Naive Bayes comparisons)

Full training code lives in `scripts/train_baseline.py`. Summary of what it does:
- 80/20 stratified train/test split (`random_state=42`)
- `TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)`
- Trains Logistic Regression (primary/saved model), Linear SVM, and Multinomial Naive Bayes for comparison
- Saves metrics table, confusion matrix, and model comparison chart to `../outputs/`
- Saves the fitted vectorizer + Logistic Regression model to `../models/` via `joblib`

In [5]:
# Re-running the training script from within the notebook for a fully reproducible, end-to-end run
!python3 ../scripts/train_baseline.py

Loaded 925 rows | class balance:
label
positive    500
negative    425
Name: count, dtype: int64

=== Logistic Regression ===
              precision    recall  f1-score   support

    negative       0.94      0.86      0.90        85
    positive       0.89      0.95      0.92       100

    accuracy                           0.91       185
   macro avg       0.91      0.90      0.91       185
weighted avg       0.91      0.91      0.91       185

=== Linear SVM ===
              precision    recall  f1-score   support

    negative       0.89      0.86      0.87        85
    positive       0.88      0.91      0.90       100

    accuracy                           0.89       185
   macro avg       0.89      0.88      0.89       185
weighted avg       0.89      0.89      0.89       185

=== Multinomial Naive Bayes ===
              precision    recall  f1-score   support

    negative       0.95      0.84      0.89        85
    positive       0.87      0.96      0.91       100

    a

### Confusion matrix — Logistic Regression (primary saved model)

![Confusion Matrix](../outputs/confusion_matrix.png)

## 4. Advanced substitute: Lexicon rule-based scorer

See `scripts/lexicon_model.py`. This plays the role of an off-the-shelf pretrained model (e.g. a Hugging Face `distilbert-base-uncased-finetuned-sst-2-english` pipeline) since it is **not trained on this dataset's labels** — it only uses general positive/negative word polarity plus negation handling. If network access were available, the only change needed would be:
```python
from transformers import pipeline
clf = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
clf(list_of_texts)
```

In [6]:
!python3 ../scripts/evaluate_advanced.py

Lexicon (advanced substitute) model -> accuracy=0.924 f1_macro=0.923

Saved updated metrics table and plots to ../outputs/


### Confusion matrix — Lexicon rule-based (advanced substitute)

![Confusion Matrix Advanced](../outputs/confusion_matrix_advanced.png)

## 5. Metrics table & model comparison

In [7]:
metrics_df = pd.read_csv('../outputs/metrics_table.csv')
metrics_df

![Model Comparison](../outputs/model_comparison.png)

**Takeaways:**
- All four approaches land in a realistic 89–92% accuracy band on this deliberately non-trivial synthetic dataset (negation + mixed-sentiment noise prevent a trivial 99%+ result).
- The lexicon-based 'advanced' substitute is competitive here because its hand-built vocabulary overlaps with the synthetic phrase bank; on truly novel, unseen review text a corpus-trained model (or a real pretrained transformer) would be expected to generalize better than a fixed lexicon.
- Logistic Regression is the strongest of the three trained baselines and is the model saved for reuse.

## 6. Saved model & inference demo

The fitted TF-IDF vectorizer and Logistic Regression model are saved to `../models/` via `joblib`. `scripts/inference.py` provides a reusable CLI: `python3 inference.py "some review text"`.

In [8]:
from inference import load_model, predict
vectorizer, model = load_model()
demo_texts = [
    'This laptop is amazing, works flawlessly and battery life is great.',
    'Total waste of money, broke after two days and support was useless.',
    'This hotel was not really the worst experience, actually quite nice.'
]
for text, label, conf in predict(demo_texts, vectorizer, model):
    print(f'[{label.upper():>8} | {conf:.1%}]  {text}')

[POSITIVE |  91.4%]  This laptop is amazing, works flawlessly and battery life is great.
[NEGATIVE |  95.7%]  Total waste of money, broke after two days and support was useless.
[POSITIVE |  62.8%]  This hotel was not really the worst experience, actually quite nice.


## Deliverables checklist
- [x] Training notebook/script — this notebook + `scripts/train_baseline.py`
- [x] Saved model or inference script — `models/*.joblib` + `scripts/inference.py`
- [x] Metrics table + confusion matrix image — `outputs/metrics_table.csv`, `outputs/confusion_matrix.png`, `outputs/confusion_matrix_advanced.png`
- [x] Documentation of offline substitutions — this notebook + `README.md`